# Data Specialist take-home assessment

Charles Maughan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
result = 2 + 2
print(result)

In [ ]:
result = 2 + 3
print(result)

## Question 12

In [ ]:
df = pd.read_excel('Annex 1 - REACH TEST DATA.xlsm', sheet_name='Annex  1 - REACH TEST DATA')

df['handwashing_check'] = (
    (df['handwashingfull'] == 'No Handwashing facility')
    & (df['Frequency.respondant.report.handwhashing.a.day'] == '7 times and more')
)

print(df['handwashing_check'].value_counts())

df.to_excel('Annex 1 - REACH TEST DATA_with_handwashing_check.xlsx', sheet_name='Annex  1 - REACH TEST DATA', index=False)

## Question 13

In [ ]:
cleaning_log = pd.read_excel('cleaninglog.xlsm', sheet_name='cleaninglog')

for _, row in cleaning_log.iterrows():
    mask = df['InterviewID'] == row['InterviewID']
    df.loc[mask, row['variable']] = row['new.value']

print(df.shape)

df.to_excel('Annex 1 - REACH TEST DATA_clean.xlsx', sheet_name='Annex  1 - REACH TEST DATA', index=False)

## Question 14

In [ ]:
house_type_summary = (
    df.dropna(subset=['house_type'])
    .groupby(['data_collection_round', 'house_type'])
    .size()
    .reset_index(name='n')
)
house_type_summary['prop'] = (
    house_type_summary['n']
    / house_type_summary.groupby('data_collection_round')['n'].transform('sum')
)

print(house_type_summary.shape)
house_type_summary

## Question 15

In [ ]:
def check_outliers(column_to_check):
    column_to_check = pd.Series(column_to_check)
    mean = column_to_check.mean()
    std = column_to_check.std()
    lower_bound = mean - 3 * std
    upper_bound = mean + 3 * std
    return (column_to_check < lower_bound) | (column_to_check > upper_bound)

## Question 16

In [ ]:
village = pd.read_excel('village.xlsm', sheet_name='village')

df = df.merge(village[['uuid', 'village']], left_on='InterviewID', right_on='uuid', how='left')
df = df.drop(columns=['uuid'])

print(df.shape)
print(df['village'].isna().sum(), 'rows missing a village')

df.to_excel('Annex 1 - REACH TEST DATA_clean.xlsx', sheet_name='Annex  1 - REACH TEST DATA', index=False)

## Question 17

In [ ]:
handwashing_order = [
    'Handwashing facility with Water & Soap',
    'Handwashing facility with Water without Soap',
    'Handwashing facility without Water and Soap',
    'No Handwashing facility',
]
handwashing_colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']

handwashing_pct = (
    df.groupby('data_collection_round')['handwashingfull']
    .value_counts(normalize=True)
    .mul(100)
    .rename('percent')
    .reset_index()
    .pivot(index='data_collection_round', columns='handwashingfull', values='percent')
    .reindex(index=['Baseline', 'End-line'], columns=handwashing_order)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(6, 5))
bottom = np.zeros(len(handwashing_pct))
for category, color in zip(handwashing_order, handwashing_colors):
    values = handwashing_pct[category].values
    ax.bar(handwashing_pct.index, values, bottom=bottom, color=color,
           edgecolor='#fcfcfb', linewidth=2, label=category)
    for x, (v, b) in enumerate(zip(values, bottom)):
        if v >= 5:
            ax.text(x, b + v / 2, f'{v:.0f}%', ha='center', va='center', color='white', fontsize=9)
    bottom += values

ax.set_xlabel('Data collection round')
ax.set_ylabel('Percent')
ax.set_ylim(0, 100)
ax.set_title('Handwashing facility type by round')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Question 18

In [ ]:
na_counts = list(map(lambda column: df[column].isna().sum(), df.columns))

print(len(na_counts))
na_counts